## BUSN9165 -- Big Data Analytics and Visualisation <br> Seminar 5

In [ ]:
# Download Java and Spark

!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q http://archive.apache.org/dist/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark

In [ ]:
# Set up the paths

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.2.1-bin-hadoop3.2"

In [ ]:
# Create a Spark session

import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better
spark.conf.set("spark.sql.caseSensitive", True) # Avoid error "Found duplicate column(s) in the data schema"
spark

## 0.&nbsp;Exercises solutions for Seminar 3

### <font color=blue>Exercise 3.0</font>

### <font color=blue>Use groupBy to show the number of tweets per sentiment category (i.e., positive, negative, neutral)</font>

In [ ]:
# First upload 'Airline-Sentiment.csv' to the runtime.

# Load the csv into a dataframe
myDF = spark.read.json("/content/Grocery_and_Gourmet_Food_5.json.gz")
myDF

# Count number of tweets per sentiment category

#tweetsPerSent = myDF.groupBy("overall").count()
#tweetsPerSent.show()

asin,image,overall,reviewText,reviewTime,reviewerID,reviewerName,style,summary,unixReviewTime,verified,vote
4639725183,null,5.0,No adverse comment.,"11 19, 2014",A1QVBUH9E1V6I8,Jamshed Mathur,null,Five Stars,1416355200,true,null
4639725183,null,5.0,Gift for college ...,"10 13, 2016",A3GEOILWLK86XM,itsjustme,null,Great product.,1476316800,true,null
4639725183,null,5.0,If you like stron...,"11 21, 2015",A32RD6L701BIGP,Krystal Clifton,null,Strong,1448064000,true,null
4639725183,null,5.0,Love the tea. The...,"08 12, 2015",A2UY1O1FBGKIE6,U. Kane,null,Great tea,1439337600,true,null
4639725183,null,5.0,I have searched e...,"05 28, 2015",A3QHVBQYDV7Z6U,The Nana,null,This is the tea I...,1432771200,true,null
4639725183,null,4.0,Tea made with Lip...,"05 9, 2015",A14MJZP7H1KHEX,Carol Ann Nix,null,Four Stars,1431129600,true,null
4639725183,null,5.0,I love this tea! ...,"05 7, 2015",A32CQJQBV7YRT,Corsair174,null,Love this tea!,1430956800,true,null
4639725183,null,5.0,Discovered this t...,"01 28, 2015",A2EUMXCQHCP25R,Metajohn,null,Great tea,1422403200,true,null
4639725183,null,4.0,Well I bought thi...,"12 23, 2014",A3QD1PUOO5I94A,B,null,Well I bought thi...,1419292800,true,null
4639725183,null,5.0,We really like th...,"12 17, 2014",A9E9L159FFMHP,S. Wood,null,We really like th...,1418774400,true,null


In [ ]:
from pyspark.sql import functions as f
myData = myDF.withColumn("sentiment", f.when((f.col("overall")>=4),"positive").when((f.col("overall")<=2),"negative"))
myData

asin,image,overall,reviewText,reviewTime,reviewerID,reviewerName,style,summary,unixReviewTime,verified,vote,sentiment
4639725183,null,5.0,No adverse comment.,"11 19, 2014",A1QVBUH9E1V6I8,Jamshed Mathur,null,Five Stars,1416355200,true,null,positive
4639725183,null,5.0,Gift for college ...,"10 13, 2016",A3GEOILWLK86XM,itsjustme,null,Great product.,1476316800,true,null,positive
4639725183,null,5.0,If you like stron...,"11 21, 2015",A32RD6L701BIGP,Krystal Clifton,null,Strong,1448064000,true,null,positive
4639725183,null,5.0,Love the tea. The...,"08 12, 2015",A2UY1O1FBGKIE6,U. Kane,null,Great tea,1439337600,true,null,positive
4639725183,null,5.0,I have searched e...,"05 28, 2015",A3QHVBQYDV7Z6U,The Nana,null,This is the tea I...,1432771200,true,null,positive
4639725183,null,4.0,Tea made with Lip...,"05 9, 2015",A14MJZP7H1KHEX,Carol Ann Nix,null,Four Stars,1431129600,true,null,positive
4639725183,null,5.0,I love this tea! ...,"05 7, 2015",A32CQJQBV7YRT,Corsair174,null,Love this tea!,1430956800,true,null,positive
4639725183,null,5.0,Discovered this t...,"01 28, 2015",A2EUMXCQHCP25R,Metajohn,null,Great tea,1422403200,true,null,positive
4639725183,null,4.0,Well I bought thi...,"12 23, 2014",A3QD1PUOO5I94A,B,null,Well I bought thi...,1419292800,true,null,positive
4639725183,null,5.0,We really like th...,"12 17, 2014",A9E9L159FFMHP,S. Wood,null,We really like th...,1418774400,true,null,positive


## 2.&nbsp;PySpark MLlib sentiment analysis

In [ ]:
myData = (myData
          #Remove handles
          .withColumn("reviewText", f.regexp_replace(f.col("reviewText"), "@[\w]*", ""))
          #Remove special characters
          .withColumn("reviewText", f.regexp_replace(f.col("reviewText"), "[^a-zA-Z']", " "))
          #Remove leading and trailing whitespaces
          .withColumn("reviewText", f.trim(f.col("reviewText")))
          #Restrict the length of the string
          .filter(f.length("reviewText")>50)
          )

In [ ]:
# Get the positive ones
myDataPos = myData.filter("sentiment = 'positive'")

# Get the negative ones
myDataNeg = myData.filter("sentiment = 'negative'")

In [ ]:
# Get a random sample from positive
myDataPosSample = myDataPos.sample(fraction=1000/myDataPos.count(), seed=9165)

# Get a random sample from negative
myDataNegSample = myDataNeg.sample(fraction=1000/myDataNeg.count(), seed=9165)

In [ ]:
# Combine into a single sample
mySample = myDataPosSample.union(myDataNegSample)

In [ ]:
# Take a look

mySample.groupBy("sentiment").count()

sentiment,count
positive,1005
negative,1003


In [ ]:
# Make a split

(training, test) = mySample.randomSplit([0.8, 0.2],seed = 9165)

In [ ]:
# Check the training

training.groupBy("sentiment").count()

sentiment,count
positive,797
negative,812


In [ ]:
myData.select('reviewText').show(truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|reviewText                                                                                    

In [ ]:
# Count number of individual rating per sentiment category

tweetsPerSent = myDF.groupBy("overall").count()
tweetsPerSent.show()

+-------+------+
|overall| count|
+-------+------+
|    1.0| 49864|
|    4.0|150771|
|    3.0| 80706|
|    2.0| 42132|
|    5.0|820387|
+-------+------+



### <font color=blue>Exercise 3.1</font>

### 2.0&nbsp;Bag-of-words approach

In [ ]:
# Use nickname feat for the subpackage
import pyspark.ml.feature as feat

# We need Pipeline to streamline the workflow
from pyspark.ml import Pipeline

# Use logistic regression
from pyspark.ml.classification import LogisticRegression

In [ ]:
# Build up the pipeline/workflow

# Split the tweets into words
splitter = feat.RegexTokenizer(
    inputCol='reviewText'
    , outputCol='text_split'
    , pattern='\s+'
)

# Remove stop words
sw_remover = feat.StopWordsRemover(
    inputCol=splitter.getOutputCol()
    , outputCol='text_noSW'
)

# Count word frequency
count_vec = feat.CountVectorizer(
    inputCol=sw_remover.getOutputCol()
    , outputCol='features'
    , vocabSize=1000
)

# Prepare the target variable
label_string = feat.StringIndexer(
    inputCol = "sentiment"
    , outputCol = "label"
)

# Logistic regression model
lr = LogisticRegression(
    maxIter=100
)


# Finally set up the pipline
sentiment_pipeline_bow = Pipeline(
    stages=[
            splitter
            , sw_remover
            , count_vec
            , label_string
            , lr
            ]
)

In [ ]:
# Import an evaluator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

lrModel = sentiment_pipeline_bow.fit(training)
lr_prediction = lrModel.transform(test)
lr_prediction.select("prediction", "sentiment", "features").show(50)
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

+----------+---------+--------------------+
|prediction|sentiment|            features|
+----------+---------+--------------------+
|       1.0| positive|(1000,[8,70,90,97...|
|       0.0| positive|(1000,[2,15,29,32...|
|       1.0| positive|(1000,[9,31,51,78...|
|       1.0| positive|(1000,[0,2,5,6,33...|
|       1.0| positive|(1000,[3,5,8,11,1...|
|       1.0| positive|(1000,[11,15,16,7...|
|       1.0| positive|(1000,[4,26,34,13...|
|       1.0| positive|(1000,[27,30,42,5...|
|       1.0| positive|(1000,[0,5,6,17,2...|
|       0.0| positive|(1000,[1,2,6,8,16...|
|       0.0| positive|(1000,[2,13,19,20...|
|       1.0| positive|(1000,[0,1,3,5,6,...|
|       1.0| positive|(1000,[130,201,30...|
|       1.0| positive|(1000,[3,4,9,28,3...|
|       1.0| positive|(1000,[2,3,7,10,4...|
|       1.0| positive|(1000,[8,56,67,76...|
|       1.0| positive|(1000,[3,8,23,26,...|
|       0.0| positive|(1000,[17,18,19,3...|
|       1.0| positive|(1000,[1,7,11,16,...|
|       1.0| positive|(1000,[2,8

In [ ]:
# Report the accuracy

lr_accuracy = evaluator.evaluate(lr_prediction)
print("Accuracy of this Logistic Regression model with bag-of-words approach is %g"% (lr_accuracy))

Accuracy of this Logistic Regression model with bag-of-words approach is 0.741855


In [ ]:
# Take a detailed look

lr_prediction.show()

+----------+-----+-------+--------------------+-----------+--------------+----------------+--------------------+--------------------+--------------+--------+----+---------+--------------------+--------------------+--------------------+-----+--------------------+--------------------+----------+
|      asin|image|overall|          reviewText| reviewTime|    reviewerID|    reviewerName|               style|             summary|unixReviewTime|verified|vote|sentiment|          text_split|           text_noSW|            features|label|       rawPrediction|         probability|prediction|
+----------+-----+-------+--------------------+-----------+--------------+----------------+--------------------+--------------------+--------------+--------+----+---------+--------------------+--------------------+--------------------+-----+--------------------+--------------------+----------+
|B00014JNI0| null|    5.0|I love this honey...|10 14, 2013| AAKY5II19PFZD|            J.V.|                null|   

### 2.1&nbsp;TF-IDF approach

In [ ]:
# splitter

splitter = feat.RegexTokenizer(inputCol='reviewText', outputCol='text_split', pattern='\s+')
mySample = splitter.transform(mySample)

In [ ]:
# Take a look

mySample.show(5)

+----------+-----+-------+--------------------+-----------+--------------+--------------------+--------------------+--------------------+--------------+--------+----+---------+--------------------+
|      asin|image|overall|          reviewText| reviewTime|    reviewerID|        reviewerName|               style|             summary|unixReviewTime|verified|vote|sentiment|          text_split|
+----------+-----+-------+--------------------+-----------+--------------+--------------------+--------------------+--------------------+--------------+--------+----+---------+--------------------+
|B00008RCN8| null|    4.0|Arrived on time  ...| 10 1, 2014| AW7NVSIV8VQ18|      Lesleigh Terry|{null, null, null...|Good quality, goo...|    1412121600|    true|null| positive|[arrived, on, tim...|
|B0000A1OEF| null|    5.0|Came quickly and ...| 02 8, 2013|A380P8LHEL3DZ9|          Eve UrAlly|{ Purple, null, n...|REALLY Purple! Ju...|    1360281600|    true|null| positive|[came, quickly, a...|
|B0000CDBP

In [ ]:
# stop words remover

sw_remover = feat.StopWordsRemover(inputCol='text_split', outputCol='text_noSW')
mySample = sw_remover.transform(mySample)

In [ ]:
# Take a look

mySample.show(5)

+----------+-----+-------+--------------------+-----------+--------------+--------------------+--------------------+--------------------+--------------+--------+----+---------+--------------------+--------------------+
|      asin|image|overall|          reviewText| reviewTime|    reviewerID|        reviewerName|               style|             summary|unixReviewTime|verified|vote|sentiment|          text_split|           text_noSW|
+----------+-----+-------+--------------------+-----------+--------------+--------------------+--------------------+--------------------+--------------+--------+----+---------+--------------------+--------------------+
|B00008RCN8| null|    4.0|Arrived on time  ...| 10 1, 2014| AW7NVSIV8VQ18|      Lesleigh Terry|{null, null, null...|Good quality, goo...|    1412121600|    true|null| positive|[arrived, on, tim...|[arrived, time, w...|
|B0000A1OEF| null|    5.0|Came quickly and ...| 02 8, 2013|A380P8LHEL3DZ9|          Eve UrAlly|{ Purple, null, n...|REALLY P

In [ ]:
# count word frequency

count_vec = feat.CountVectorizer(inputCol='text_noSW', outputCol='vector', vocabSize=1000)
mySample = count_vec.fit(mySample).transform(mySample)

In [ ]:
# Take a look

mySample.show(5)

+----------+-----+-------+--------------------+-----------+--------------+--------------------+--------------------+--------------------+--------------+--------+----+---------+--------------------+--------------------+--------------------+
|      asin|image|overall|          reviewText| reviewTime|    reviewerID|        reviewerName|               style|             summary|unixReviewTime|verified|vote|sentiment|          text_split|           text_noSW|              vector|
+----------+-----+-------+--------------------+-----------+--------------+--------------------+--------------------+--------------------+--------------+--------+----+---------+--------------------+--------------------+--------------------+
|B00008RCN8| null|    4.0|Arrived on time  ...| 10 1, 2014| AW7NVSIV8VQ18|      Lesleigh Terry|{null, null, null...|Good quality, goo...|    1412121600|    true|null| positive|[arrived, on, tim...|[arrived, time, w...|(1000,[2,13,15,28...|
|B0000A1OEF| null|    5.0|Came quickly a

In [ ]:
# Calculate IDF

idf_cal = feat.IDF(inputCol='vector', outputCol='features')
mySample = idf_cal.fit(mySample).transform(mySample)

In [ ]:
# Take a look

mySample.show(5)

+----------+-----+-------+--------------------+-----------+--------------+--------------------+--------------------+--------------------+--------------+--------+----+---------+--------------------+--------------------+--------------------+--------------------+
|      asin|image|overall|          reviewText| reviewTime|    reviewerID|        reviewerName|               style|             summary|unixReviewTime|verified|vote|sentiment|          text_split|           text_noSW|              vector|            features|
+----------+-----+-------+--------------------+-----------+--------------+--------------------+--------------------+--------------------+--------------+--------+----+---------+--------------------+--------------------+--------------------+--------------------+
|B00008RCN8| null|    4.0|Arrived on time  ...| 10 1, 2014| AW7NVSIV8VQ18|      Lesleigh Terry|{null, null, null...|Good quality, goo...|    1412121600|    true|null| positive|[arrived, on, tim...|[arrived, time, w...

In [ ]:
# Create indexer for sentiment

label_string = feat.StringIndexer(inputCol='sentiment', outputCol='label')
mySample = label_string.fit(mySample).transform(mySample)

In [ ]:
# Make a split

(training, test) = mySample.randomSplit([0.8, 0.2],seed = 9165)

In [ ]:
# Use logistic regression
lr = LogisticRegression(maxIter=100)

# Set up the model
lrModel = lr.fit(training)
lr_prediction = lrModel.transform(test)
lr_prediction.select("prediction", "sentiment", "features").show(50)
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

+----------+---------+--------------------+
|prediction|sentiment|            features|
+----------+---------+--------------------+
|       0.0| positive|(1000,[8,79,92,11...|
|       1.0| positive|(1000,[2,15,27,31...|
|       1.0| positive|(1000,[9,41,50,72...|
|       0.0| positive|(1000,[0,2,4,6,32...|
|       0.0| positive|(1000,[3,4,8,11,1...|
|       0.0| positive|(1000,[11,14,15,6...|
|       0.0| positive|(1000,[5,23,33,14...|
|       0.0| positive|(1000,[29,30,38,5...|
|       0.0| positive|(1000,[0,4,6,16,2...|
|       1.0| positive|(1000,[1,2,6,8,14...|
|       1.0| positive|(1000,[2,13,17,21...|
|       0.0| positive|(1000,[0,1,3,4,6,...|
|       0.0| positive|(1000,[140,219,31...|
|       0.0| positive|(1000,[3,5,9,28,4...|
|       0.0| positive|(1000,[2,3,7,10,3...|
|       0.0| positive|(1000,[8,59,71,72...|
|       0.0| positive|(1000,[3,8,23,24,...|
|       0.0| positive|(1000,[16,19,21,3...|
|       0.0| positive|(1000,[1,7,11,14,...|
|       0.0| positive|(1000,[2,8

In [ ]:
# Report the accuracy

lr_accuracy = evaluator.evaluate(lr_prediction)
print("Accuracy of this Logistic Regression model with TF-IDF approach is %g"% (lr_accuracy))

Accuracy of this Logistic Regression model with TF-IDF approach is 0.754386


### <font color=blue>Exercise 5.0</font>

### <font color=blue>Reorganise the steps in Section 2.1 with Pipeline.</font>

In [ ]:
# Make a split with different proportion

(training, test) = mySample.randomSplit([0.6, 0.4],seed = 1234)

In [ ]:
# Use logistic regression
lr = LogisticRegression(maxIter=100)

# Set up the model
lrModel = lr.fit(training)
lr_prediction = lrModel.transform(test)
lr_prediction.select("prediction", "sentiment", "features").show(50)
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

+----------+---------+--------------------+
|prediction|sentiment|            features|
+----------+---------+--------------------+
|       0.0| positive|(1000,[2,13,15,28...|
|       0.0| positive|(1000,[50,81,180,...|
|       1.0| positive|(1000,[2,29,77,81...|
|       0.0| positive|(1000,[1,6,19,29,...|
|       0.0| positive|(1000,[8,11,12,20...|
|       0.0| positive|(1000,[9,41,50,72...|
|       0.0| positive|(1000,[3,4,8,11,1...|
|       0.0| positive|(1000,[1,21,53,96...|
|       0.0| positive|(1000,[4,5,336,56...|
|       1.0| positive|(1000,[2,64,80,90...|
|       0.0| positive|(1000,[1,9,31,32,...|
|       0.0| positive|(1000,[3,104,167,...|
|       0.0| positive|(1000,[1,6,11,12,...|
|       0.0| positive|(1000,[1,5,6,9,17...|
|       0.0| positive|(1000,[3,5,8,17,4...|
|       0.0| positive|(1000,[1,28,794],...|
|       0.0| positive|(1000,[1,4,5,16,2...|
|       0.0| positive|(1000,[29,30,38,5...|
|       0.0| positive|(1000,[5,49,62,91...|
|       1.0| positive|(1000,[0,3

In [ ]:
# Report the accuracy with ne wproportion and seed value

lr_accuracy = evaluator.evaluate(lr_prediction)
print("Accuracy of this Logistic Regression model with TF-IDF approach is %g"% (lr_accuracy))

Accuracy of this Logistic Regression model with TF-IDF approach is 0.755051


PySpark MLlib parameter tuning, cross validation, and model comparison


In [ ]:
# More operations are available from Spark's SQL functions
from pyspark.sql import functions as f

# Use nickname feat for the subpackage
import pyspark.ml.feature as feat

# We need Pipeline to streamline the workflow
from pyspark.ml import Pipeline

# Use logistic regression
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier

# Import an evaluator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Additional functions for tuning parameters
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder